# Build Paper Nodes (2022–2025 Data)

**Purpose:** Extract all unique papers from the 2022–2025 metadata JSONs and build a paper_nodes table in the same format as the existing `paper_nodes.csv`.

**Key difference from old pipeline:**
- Old data: local integer IDs → needed `mapped_json` → `03_combined_json` → title-based OpenAlex query
- New data: OpenAlex IDs already in metadata → no mapping step needed

**Output:** `paper_nodes_2022_2025.csv`

In [1]:
import json
import os
import pandas as pd
from glob import glob

In [2]:
# Paths
DATA_DIR   = "../Scientific_Novelty_Detection_2022_2025/data(2022-2025)/cache/"
OUT_DIR    = "../outputs/final/"

In [3]:
# Domain mapping: filename prefix → standard domain code
DOMAIN_MAP = {
    'dia'  : 'DIA',
    'mt'   : 'MT',
    'nli'  : 'NLI',
    'par'  : 'PAR',
    'qa'   : 'QA',
    'sa'   : 'SA',
    'sum'  : 'SUM',
    'nre'  : 'NRE',
}

# Split mapping: filename prefix → standard split code
SPLIT_MAP = {
    'dia2022_2025'    : 'NOVEL',
    'mt2022_2025'     : 'NOVEL',
    'qa2022_2025'     : 'NOVEL',
    'sa2022_2025'     : 'NOVEL',
    'sum2022_2025'    : 'NOVEL',
    'blogs_dia'       : 'BLOG',
    'blogs_mt'        : 'BLOG',
    'blogs_nli'       : 'BLOG',
    'blogs_par'       : 'BLOG',
    'blogs_qa'        : 'BLOG',
    'blogs_sa'        : 'BLOG',
    'blogs_sum'       : 'BLOG',
    'skg_dia'         : 'SKG',
    'skg_mt'          : 'SKG',
    'skg_nli'         : 'SKG',
    'skg_par'         : 'SKG',
    'skg_qa'          : 'SKG',
    'skg_sa'          : 'SKG',
    'skg_sum'         : 'SKG',
}

# Domain from filename for each file type
DOMAIN_FROM_FILE = {
    'dia2022_2025'    : 'DIA',
    'mt2022_2025'     : 'MT',
    'qa2022_2025'     : 'QA',
    'sa2022_2025'     : 'SA',
    'sum2022_2025'    : 'SUM',
    'blogs_dia'       : 'DIA',
    'blogs_mt'        : 'MT',
    'blogs_nli'       : 'NLI',
    'blogs_par'       : 'PAR',
    'blogs_qa'        : 'QA',
    'blogs_sa'        : 'SA',
    'blogs_sum'       : 'SUM',
    'skg_dia'         : 'DIA',
    'skg_mt'          : 'MT',
    'skg_nli'         : 'NLI',
    'skg_par'         : 'PAR',
    'skg_qa'          : 'QA',
    'skg_sa'          : 'SA',
    'skg_sum'         : 'SUM',
}

print("Config loaded.")

Config loaded.


In [4]:
# Load all metadata JSON files
json_files = glob(os.path.join(DATA_DIR, "*.json"))
print(f"Found {len(json_files)} metadata JSON files:")
for f in sorted(json_files):
    print(" ", os.path.basename(f))

Found 19 metadata JSON files:
  Blogs_Dia_metadata.json
  Blogs_MT_metadata.json
  Blogs_NLI_metadata.json
  Blogs_Par_metadata.json
  Blogs_QA_metadata.json
  Blogs_SA_metadata.json
  Blogs_Sum_metadata.json
  Dia2022_2025_metadata.json
  MT2022_2025_metadata.json
  QA2022_2025_metadata.json
  SA2022_2025_metadata.json
  SKG_Dia_metadata.json
  SKG_MT_metadata.json
  SKG_NLI_metadata.json
  SKG_Par_metadata.json
  SKG_QA_metadata.json
  SKG_SA_metadata.json
  SKG_Sum_metadata.json
  Sum2022_2025_metadata.json


In [5]:
def extract_openalex_id(url):
    return url.strip().rstrip('/').split('/')[-1]


def get_file_key(filename):
    name = os.path.basename(filename).replace('_metadata.json', '').lower()
    return name


papers = []
skipped_files = []

for filepath in sorted(json_files):
    file_key = get_file_key(filepath)

    if file_key not in SPLIT_MAP:
        print(f"  ⚠ No mapping for '{file_key}' — skipping {os.path.basename(filepath)}")
        skipped_files.append(filepath)
        continue

    split  = SPLIT_MAP[file_key]
    domain = DOMAIN_FROM_FILE[file_key]

    with open(filepath, encoding='utf-8') as f:
        data = json.load(f)

    for entry in data:
        openalex_id = extract_openalex_id(entry['id'])
        global_id   = f"{split}_{domain}_{openalex_id}"

        papers.append({
            'node_id'        : global_id,
            'node_type'      : 'Paper',
            'title'          : entry.get('title', ''),
            'domain'         : domain,
            'split'          : split,
            'year'           : entry.get('year', None),
            'openalex_id'    : openalex_id,
            'cited_by_count' : entry.get('cited_by_count', None),
            'score'          : 1.0,       # direct OpenAlex pull — no matching score needed
            'method'         : 'openalex_direct',
        })

    print(f"  ✓ {os.path.basename(filepath):45s} split={split:6s} domain={domain:4s} papers={len(data)}")


new_pn = pd.DataFrame(papers)
print(f"\nTotal rows before dedup: {len(new_pn)}")

  ✓ Blogs_Dia_metadata.json                       split=BLOG   domain=DIA  papers=120
  ✓ Blogs_MT_metadata.json                        split=BLOG   domain=MT   papers=120
  ✓ Blogs_NLI_metadata.json                       split=BLOG   domain=NLI  papers=120
  ✓ Blogs_Par_metadata.json                       split=BLOG   domain=PAR  papers=120
  ✓ Blogs_QA_metadata.json                        split=BLOG   domain=QA   papers=120
  ✓ Blogs_SA_metadata.json                        split=BLOG   domain=SA   papers=120
  ✓ Blogs_Sum_metadata.json                       split=BLOG   domain=SUM  papers=120
  ✓ Dia2022_2025_metadata.json                    split=NOVEL  domain=DIA  papers=240
  ✓ MT2022_2025_metadata.json                     split=NOVEL  domain=MT   papers=111
  ✓ QA2022_2025_metadata.json                     split=NOVEL  domain=QA   papers=22
  ✓ SA2022_2025_metadata.json                     split=NOVEL  domain=SA   papers=56
  ✓ SKG_Dia_metadata.json                         split=

In [6]:
# Deduplicate — same OpenAlex paper may appear in multiple domain files
print("Duplicate node_ids:", new_pn['node_id'].duplicated().sum())
new_pn = new_pn.drop_duplicates(subset='node_id').reset_index(drop=True)
print(f"After dedup: {len(new_pn)} papers")
print()
print("Split breakdown:")
print(new_pn['split'].value_counts())
print()
print("Domain breakdown:")
print(new_pn['domain'].value_counts())
print()
print("Year range:", new_pn['year'].min(), '–', new_pn['year'].max())

Duplicate node_ids: 0
After dedup: 2331 papers

Split breakdown:
split
SKG      1050
BLOG      840
NOVEL     441
Name: count, dtype: int64

Domain breakdown:
domain
DIA    510
MT     381
SA     326
QA     292
SUM    282
NLI    270
PAR    270
Name: count, dtype: int64

Year range: 2022 – 2025


In [7]:
# Check overlap with existing paper_nodes
old_pn = pd.read_csv(OUT_DIR + 'paper_nodes.csv')

old_openalex_ids = set(old_pn['openalex_id'].dropna().astype(str))
new_openalex_ids = set(new_pn['openalex_id'].dropna().astype(str))

overlap = old_openalex_ids & new_openalex_ids
print(f"Old paper_nodes: {len(old_pn)}")
print(f"New papers:      {len(new_pn)}")
print(f"Overlapping OpenAlex IDs: {len(overlap)}")

if len(overlap) > 0:
    print("Sample overlapping IDs:", list(overlap)[:5])
    # Remove overlapping papers from new data — they already exist
    new_pn = new_pn[~new_pn['openalex_id'].isin(overlap)].reset_index(drop=True)
    print(f"New papers after removing overlap: {len(new_pn)}")

Old paper_nodes: 2529
New papers:      2331
Overlapping OpenAlex IDs: 0


In [8]:
# Sanity checks
assert new_pn['node_id'].is_unique, "Duplicate node_ids!"
assert new_pn['node_type'].eq('Paper').all()
assert new_pn['title'].notna().all(), "Missing titles!"
print("All sanity checks passed.")
print()
print(new_pn.head(5).to_string())

All sanity checks passed.

                node_id node_type                                                                         title domain split  year  openalex_id  cited_by_count  score           method
0  BLOG_DIA_W4309674289     Paper                        Survey of Hallucination in Natural Language Generation    DIA  BLOG  2022  W4309674289             NaN    1.0  openalex_direct
1  BLOG_DIA_W4224436908     Paper                                             Microbiota in health and diseases    DIA  BLOG  2022  W4224436908             NaN    1.0  openalex_direct
2  BLOG_DIA_W4384918448     Paper                           Llama 2: Open Foundation and Fine-Tuned Chat Models    DIA  BLOG  2023  W4384918448             NaN    1.0  openalex_direct
3  BLOG_DIA_W1994546180     Paper                                                  Giving an Account of Oneself    DIA  BLOG  2025  W1994546180             NaN    1.0  openalex_direct
4  BLOG_DIA_W4366420437     Paper  What Is the Impact

In [9]:
# Save
new_pn.to_csv(OUT_DIR + 'paper_nodes_2022_2025.csv', index=False)
print(f"Saved: paper_nodes_2022_2025.csv  ({len(new_pn)} papers)")

Saved: paper_nodes_2022_2025.csv  (2331 papers)


In [10]:
new_pn.shape

(2331, 10)

In [11]:
new_pn.head()

,node_id,node_type,title,domain,split,year,openalex_id,cited_by_count,score,method
0,BLOG_DIA_W4309674289,Paper,Survey of Hallucination in Natural Language Ge...,DIA,BLOG,2022,W4309674289,NaN,1.0,openalex_direct
1,BLOG_DIA_W4224436908,Paper,Microbiota in health and diseases,DIA,BLOG,2022,W4224436908,NaN,1.0,openalex_direct
2,BLOG_DIA_W4384918448,Paper,Llama 2: Open Foundation and Fine-Tuned Chat M...,DIA,BLOG,2023,W4384918448,NaN,1.0,openalex_direct
3,BLOG_DIA_W1994546180,Paper,Giving an Account of Oneself,DIA,BLOG,2025,W1994546180,NaN,1.0,openalex_direct
4,BLOG_DIA_W4366420437,Paper,What Is the Impact of ChatGPT on Education? A ...,DIA,BLOG,2023,W4366420437,NaN,1.0,openalex_direct
